# Almost replaced by add_director_stats, outside of specifically the encoding in here. Will also add this to director stats later.

In [1]:
import pandas as pd
import re
import unicodedata
from sklearn.preprocessing import MultiLabelBinarizer
import ast

In [2]:
df = pd.read_csv('initial_df.csv')
df_director_info = pd.read_csv('directors.csv')

In [3]:
df.head()

,title,release_year,personal_rating,avg_rating,genre,director,country,language
0,Oppenheimer,2023,4.5,4.17,"['Drama', 'History']",Christopher Nolan,"['UK', 'USA']","['English', 'Dutch', 'English']"
1,Spider-Man: Across the Spider-Verse,2023,5.0,4.41,"['Animation', 'Science Fiction', 'Adventure', ...","Kemp Powers, Justin K. Thompson et al",['USA'],"['English', 'English', 'Hindi', 'Italian', 'Sp..."
2,The Super Mario Bros. Movie,2023,4.0,3.22,"['Adventure', 'Animation', 'Comedy', 'Family',...","Aaron Horvath, Michael Jelenic","['Japan', 'USA']",['English']
3,Avatar: The Way of Water,2022,3.0,3.62,"['Science Fiction', 'Action', 'Adventure']",James Cameron,['USA'],['English']
4,Jujutsu Kaisen 0,2021,4.0,3.93,"['Action', 'Animation', 'Fantasy']",Sunghoo Park,['Japan'],['Japanese']


In [4]:
df_director_info.head()

,director,number_of_movies,average_movie_score,average_last_5
0,Christopher Nolan,18,7.112333,7.427000
1,Kemp Powers,2,8.220000,8.220000
2,Justin K. Thompson,1,8.338000,8.338000
3,Aaron Horvath,3,8.257667,8.257667
4,Michael Jelenic,2,8.800500,8.800500


In [5]:
# on letterbox you rate out of 5, while on movie database ratings are out of 10. So have to half the values.
def half_column_value(val):
    return val/2
df_director_info['average_movie_score'] = df_director_info['average_movie_score'].apply(half_column_value)
df_director_info['average_last_5'] = df_director_info['average_last_5'].apply(half_column_value)

In [7]:
df_director_info.head()

,director,number_of_movies,average_movie_score,average_last_5
0,Christopher Nolan,18,3.556167,3.713500
1,Kemp Powers,2,4.110000,4.110000
2,Justin K. Thompson,1,4.169000,4.169000
3,Aaron Horvath,3,4.128833,4.128833
4,Michael Jelenic,2,4.400250,4.400250


In [8]:
def seperate_directors(s):
    if pd.isna(s):
        return s
    
    # Remove 'et al' (case-insensitive, optional period)
    s = re.sub(r'\bet\s+al\.?\b', '', s, flags=re.IGNORECASE)
    
    # Remove extra commas caused by removal
    s = re.sub(r',\s*,', ',', s)
    
    # Strip leading/trailing commas and whitespace
    s = s.strip(' ,')
    
    return s

In [9]:
df['director'] = df['director'].apply(seperate_directors)

In [10]:
df.head()

,title,release_year,personal_rating,avg_rating,genre,director,country,language
0,Oppenheimer,2023,4.5,4.17,"['Drama', 'History']",Christopher Nolan,"['UK', 'USA']","['English', 'Dutch', 'English']"
1,Spider-Man: Across the Spider-Verse,2023,5.0,4.41,"['Animation', 'Science Fiction', 'Adventure', ...","Kemp Powers, Justin K. Thompson",['USA'],"['English', 'English', 'Hindi', 'Italian', 'Sp..."
2,The Super Mario Bros. Movie,2023,4.0,3.22,"['Adventure', 'Animation', 'Comedy', 'Family',...","Aaron Horvath, Michael Jelenic","['Japan', 'USA']",['English']
3,Avatar: The Way of Water,2022,3.0,3.62,"['Science Fiction', 'Action', 'Adventure']",James Cameron,['USA'],['English']
4,Jujutsu Kaisen 0,2021,4.0,3.93,"['Action', 'Animation', 'Fantasy']",Sunghoo Park,['Japan'],['Japanese']


In [11]:
df_exploded = (df.assign(director=df['director'].str.split(",")).explode('director'))

In [12]:
df_exploded.head()

,title,release_year,personal_rating,avg_rating,genre,director,country,language
0,Oppenheimer,2023,4.5,4.17,"['Drama', 'History']",Christopher Nolan,"['UK', 'USA']","['English', 'Dutch', 'English']"
1,Spider-Man: Across the Spider-Verse,2023,5.0,4.41,"['Animation', 'Science Fiction', 'Adventure', ...",Kemp Powers,['USA'],"['English', 'English', 'Hindi', 'Italian', 'Sp..."
1,Spider-Man: Across the Spider-Verse,2023,5.0,4.41,"['Animation', 'Science Fiction', 'Adventure', ...",Justin K. Thompson,['USA'],"['English', 'English', 'Hindi', 'Italian', 'Sp..."
2,The Super Mario Bros. Movie,2023,4.0,3.22,"['Adventure', 'Animation', 'Comedy', 'Family',...",Aaron Horvath,"['Japan', 'USA']",['English']
2,The Super Mario Bros. Movie,2023,4.0,3.22,"['Adventure', 'Animation', 'Comedy', 'Family',...",Michael Jelenic,"['Japan', 'USA']",['English']


In [13]:
def normalize_title(s: str) -> str:
    # 1. Normalize Unicode (NFKC handles compatibility chars)
    s = unicodedata.normalize("NFKC", s)

    # 2. Lowercase
    s = s.lower()

    # 3. Replace all Unicode whitespace with a normal space
    s = re.sub(r"\s+", " ", s)

    # 4. Normalize dash variants (after NFKC, many are already unified)
    s = s.replace("–", "-").replace("—", "-")

    # 5. Strip leading/trailing space
    return s.strip()

In [14]:
df_exploded['director'] = df_exploded['director'].apply(normalize_title)
df_director_info['director'] = df_director_info['director'].apply(normalize_title)

In [15]:
df_merged = df_exploded.merge(
    df_director_info,
    left_on='director',
    right_on='director',
    how='left'
)

In [16]:
df_merged.head(12)

,title,release_year,personal_rating,avg_rating,genre,director,country,language,number_of_movies,average_movie_score,average_last_5
0,Oppenheimer,2023,4.5,4.17,"['Drama', 'History']",christopher nolan,"['UK', 'USA']","['English', 'Dutch', 'English']",18.0,3.556167,3.713500
1,Spider-Man: Across the Spider-Verse,2023,5.0,4.41,"['Animation', 'Science Fiction', 'Adventure', ...",kemp powers,['USA'],"['English', 'English', 'Hindi', 'Italian', 'Sp...",2.0,4.110000,4.110000
2,Spider-Man: Across the Spider-Verse,2023,5.0,4.41,"['Animation', 'Science Fiction', 'Adventure', ...",justin k. thompson,['USA'],"['English', 'English', 'Hindi', 'Italian', 'Sp...",1.0,4.169000,4.169000
3,The Super Mario Bros. Movie,2023,4.0,3.22,"['Adventure', 'Animation', 'Comedy', 'Family',...",aaron horvath,"['Japan', 'USA']",['English'],3.0,4.128833,4.128833
4,The Super Mario Bros. Movie,2023,4.0,3.22,"['Adventure', 'Animation', 'Comedy', 'Family',...",michael jelenic,"['Japan', 'USA']",['English'],2.0,4.400250,4.400250
5,Avatar: The Way of Water,2022,3.0,3.62,"['Science Fiction', 'Action', 'Adventure']",james cameron,['USA'],['English'],15.0,3.505800,3.565500
6,Jujutsu Kaisen 0,2021,4.0,3.93,"['Action', 'Animation', 'Fantasy']",sunghoo park,['Japan'],['Japanese'],3.0,3.478667,3.478667
7,Spider-Man: No Way Home,2021,4.0,3.83,"['Adventure', 'Action', 'Science Fiction']",jon watts,['USA'],"['English', 'English', 'Tagalog']",11.0,3.411500,3.608400
8,No Time to Die,2021,3.0,3.52,"['Action', 'Adventure', 'Thriller']",cary joji fukunaga,"['UK', 'USA']","['English', 'Spanish', 'French', 'Russian', 'E...",6.0,3.511750,3.594100
9,Dune,2021,3.5,3.88,"['Science Fiction', 'Adventure']",denis villeneuve,['USA'],"['English', 'Chinese', 'English']",17.0,3.406265,3.853400


In [17]:
df_merged[df_merged.isna().any(axis=1)]

,title,release_year,personal_rating,avg_rating,genre,director,country,language,number_of_movies,average_movie_score,average_last_5
10,JUJUTSU KAISEN,2020,3.5,4.22,['Animation'],yui umemoto,['Japan'],['Japanese'],NaN,NaN,NaN
11,JUJUTSU KAISEN,2020,3.5,4.22,['Animation'],ryohei takeshita,['Japan'],['Japanese'],NaN,NaN,NaN


In [18]:
df_merged[df_merged['title'] == 'JUJUTSU KAISEN'] # this result makes sense since this is an anime, not a movie, so wasnt added

,title,release_year,personal_rating,avg_rating,genre,director,country,language,number_of_movies,average_movie_score,average_last_5
10,JUJUTSU KAISEN,2020,3.5,4.22,['Animation'],yui umemoto,['Japan'],['Japanese'],NaN,NaN,NaN
11,JUJUTSU KAISEN,2020,3.5,4.22,['Animation'],ryohei takeshita,['Japan'],['Japanese'],NaN,NaN,NaN


In [19]:
df_clean = df_merged.dropna(subset='number_of_movies') # only drop if there are na values in one of the imported rows
df_clean

,title,release_year,personal_rating,avg_rating,genre,director,country,language,number_of_movies,average_movie_score,average_last_5
0,Oppenheimer,2023,4.5,4.17,"['Drama', 'History']",christopher nolan,"['UK', 'USA']","['English', 'Dutch', 'English']",18.0,3.556167,3.713500
1,Spider-Man: Across the Spider-Verse,2023,5.0,4.41,"['Animation', 'Science Fiction', 'Adventure', ...",kemp powers,['USA'],"['English', 'English', 'Hindi', 'Italian', 'Sp...",2.0,4.110000,4.110000
2,Spider-Man: Across the Spider-Verse,2023,5.0,4.41,"['Animation', 'Science Fiction', 'Adventure', ...",justin k. thompson,['USA'],"['English', 'English', 'Hindi', 'Italian', 'Sp...",1.0,4.169000,4.169000
3,The Super Mario Bros. Movie,2023,4.0,3.22,"['Adventure', 'Animation', 'Comedy', 'Family',...",aaron horvath,"['Japan', 'USA']",['English'],3.0,4.128833,4.128833
4,The Super Mario Bros. Movie,2023,4.0,3.22,"['Adventure', 'Animation', 'Comedy', 'Family',...",michael jelenic,"['Japan', 'USA']",['English'],2.0,4.400250,4.400250
...,...,...,...,...,...,...,...,...,...,...,...
125,E.T. the Extra-Terrestrial,1982,2.0,3.84,"['Science Fiction', 'Family', 'Fantasy', 'Adve...",steven spielberg,['USA'],['English'],42.0,3.494500,3.541100
126,The Empire Strikes Back,1980,3.5,4.40,"['Science Fiction', 'Adventure', 'Action']",irvin kershner,['USA'],['English'],16.0,2.876875,2.906700
127,Star Wars,1977,4.0,4.16,"['Adventure', 'Science Fiction', 'Action']",george lucas,['USA'],['English'],20.0,3.310475,3.610700
128,The Godfather Part II,1974,3.5,4.59,"['Drama', 'Crime']",francis ford coppola,['USA'],"['English', 'English', 'Italian', 'Latin', 'Sp...",38.0,3.109961,2.707900


In [20]:
df_clean = df_clean.rename(columns={
    'number_of_movies': 'director_avg_num_movies',
    'average_movie_score': 'director_avg_movie_score',
    'average_last_5': 'director_avg_last_5_movies'
             }
)

In [21]:
mean_cols = [
    "director_avg_num_movies",
    "director_avg_movie_score",
    "director_avg_last_5_movies"
]

other_cols = [c for c in df_clean.columns if c not in mean_cols]
agg_dict = {c: 'first' for c in other_cols}
agg_dict.update({c: 'mean' for c in mean_cols})
agg_dict['director'] = list
# agg_dict['director'] = lambda x: list(dict.fromkeys(x))

In [22]:
df_avg = df_clean.groupby('title', as_index=False).agg(agg_dict)
df_avg["director1"] = df_avg["director"].apply(lambda x: x[0] if len(x) > 0 else None)
df_avg["director2"] = df_avg["director"].apply(lambda x: x[1] if len(x) > 1 else None)

In [23]:
df_avg.drop('director', axis=1, inplace=True)

In [24]:
df_avg.head(15)

,title,release_year,personal_rating,avg_rating,genre,country,language,director_avg_num_movies,director_avg_movie_score,director_avg_last_5_movies,director1,director2
0,2 Fast 2 Furious,2003,2.5,3.15,"['Crime', 'Action', 'Thriller']","['Germany', 'USA']","['English', 'English', 'Spanish']",12.0,3.358417,3.0871,john singleton,None
1,2012,2009,2.0,2.55,"['Adventure', 'Action', 'Science Fiction']",['USA'],"['English', 'Russian', 'Hindi', 'German', 'Ita...",19.0,3.072658,3.1191,roland emmerich,None
2,American Beauty,1999,0.5,3.95,['Drama'],['USA'],['English'],15.0,3.642567,3.8378,sam mendes,None
3,American Psycho,2000,3.5,3.79,"['Crime', 'Drama', 'Thriller', 'Horror']","['Canada', 'USA']","['English', 'English', 'Spanish', 'Cantonese']",8.0,3.059500,2.9130,mary harron,None
4,Another Round,2020,4.0,4.09,"['Drama', 'Comedy']","['Denmark', 'Netherlands', 'Sweden']","['Danish', 'Danish', 'Swedish']",16.0,3.362938,3.5784,thomas vinterberg,None
5,Avatar,2009,3.5,3.69,"['Fantasy', 'Adventure', 'Science Fiction', 'A...","['USA', 'UK']","['English', 'English', 'Spanish']",15.0,3.505800,3.5655,james cameron,None
6,Avatar: The Way of Water,2022,3.0,3.62,"['Science Fiction', 'Action', 'Adventure']",['USA'],['English'],15.0,3.505800,3.5655,james cameron,None
7,Avengers: Age of Ultron,2015,2.0,3.21,"['Science Fiction', 'Action', 'Adventure']",['USA'],['English'],8.0,3.157125,3.1047,joss whedon,None
8,Avengers: Endgame,2019,3.0,3.94,"['Science Fiction', 'Action', 'Adventure']",['USA'],"['English', 'English', 'Japanese', 'Xhosa']",10.0,3.351050,3.7192,joe russo,anthony russo
9,Avengers: Infinity War,2018,5.0,4.02,"['Action', 'Science Fiction', 'Adventure']",['USA'],"['English', 'English', 'Xhosa']",10.0,3.351050,3.7192,anthony russo,joe russo


In [25]:
df_avg.isna().sum()

title                           0
release_year                    0
personal_rating                 0
avg_rating                      0
genre                           0
country                         0
language                        0
director_avg_num_movies         0
director_avg_movie_score        0
director_avg_last_5_movies      0
director1                       0
director2                     102
dtype: int64

In [26]:
print(type(df_avg['genre'].iloc[0]))

<class 'str'>


In [27]:
# need to convert the string of list of strings to just a list of strings to do encoding
cat_features = ['genre', 'country', 'language']

for column in cat_features:
    df_avg[column] = df_avg[column].apply(lambda x: ast.literal_eval(x))

In [28]:
def encode_category(dframe, column_name, name_of_new_column):

    mlb = MultiLabelBinarizer()
    column_encoded = mlb.fit_transform(dframe[column_name])
    new_dframe = pd.DataFrame(column_encoded, columns=[f"{column_name}: {g}" for g in mlb.classes_], index=dframe.index)
    # columns=[f"Genre: {g}" for g in mlb.classes_], 
    new_dframe[name_of_new_column] = 0

    for category in new_dframe.columns:
        if new_dframe[category].sum() < 2: # if rare category
            new_dframe[name_of_new_column] = new_dframe[name_of_new_column] | new_dframe[category] # 1 in same row if 1 already in new column or 1 in category, else 0
            new_dframe = new_dframe.drop(category, axis=1) # drop the rare category
    
    dframe = dframe.drop(column_name, axis=1)

    return pd.concat([dframe, new_dframe], axis=1)

In [29]:
df_encoded = df_avg[:]
for column in cat_features:
    df_encoded = encode_category(df_encoded, column, f'rare {column}')
df_encoded.head(2)

,title,release_year,personal_rating,avg_rating,director_avg_num_movies,director_avg_movie_score,director_avg_last_5_movies,director1,director2,genre: Action,...,language: Italian,language: Japanese,language: Latin,language: Portuguese,language: Russian,language: Spanish,language: Swedish,language: Urdu,language: Xhosa,rare language
0,2 Fast 2 Furious,2003,2.5,3.15,12.0,3.358417,3.0871,john singleton,None,1,...,0,0,0,0,0,1,0,0,0,0
1,2012,2009,2.0,2.55,19.0,3.072658,3.1191,roland emmerich,None,1,...,1,0,1,1,1,1,0,0,0,1


In [30]:
df_avg.to_csv('all_data.csv', index=False)

In [31]:
df_encoded.to_csv('all_data_encoded.csv', index=False)

Now the same for unseen data. Instead, just remake all of the above into a big function you can apply to a dataset